# P236: Evaluating an SQL Agent end-to-end Performance

https://github.com/langchain-ai/learning-langchain/blob/master/ch10/py/create_sql_dataset.py

In [ ]:
# # create and upload a datset with answer to langsmith

# from dotenv import load_dotenv
# from langsmith import Client

# load_dotenv()

# client = Client()

# DATASET_ID = "6009592b-7361-4ac5-b6ef-93f852f064ed" #get it from the dataset tab from langsmith

# examples = [
#     ("Which country's customers spent the most? And how much did they spend?",
#      "The country whose customers spent the most is the USA, with a total expenditure of $523.06"),
#     ("What was the most purchased track of 2013?",
#      "The most purchased track of 2013 was Hot Girl."),
#     ("How many albums does the artist Led Zeppelin have?",
#      "Led Zeppelin has 14 albums"),
#     ("What is the total price for the album “Big Ones”?",
#      "The total price for the album 'Big Ones' is 14.85"),
#     ("Which sales agent made the most in sales in 2009?",
#      "Steve Johnson made the most sales in 2009"),
# ]

# formatted_examples = [
#     {
#         "inputs": {"input": question},
#         "outputs": {"output": answer},
#         "metadata": {"source": "langchain blogs"},
#     }
#     for question, answer in examples
# ]

# dataset = client.read_dataset(dataset_id=DATASET_ID)

# client.create_examples(
#     dataset_id=dataset.id,
#     examples=formatted_examples
# )

# print(f"Added {len(formatted_examples)} examples to dataset: {dataset.id}")
# print(dataset.url)

Added 5 examples to dataset: 6009592b-7361-4ac5-b6ef-93f852f064ed
https://smith.langchain.com/o/6d4bdaec-72d5-480a-8ecb-27962e000fe7/datasets/6009592b-7361-4ac5-b6ef-93f852f064ed


In [ ]:
# import chinook.db file
# !curl -s https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql | sqlite3 Chinook.db

# from langchain_community.utilities import SQLDatabase

# db = SQLDatabase.from_uri("sqlite:///Chinook.db")

## P:236 testing an agent final response

In [11]:
from my_agent.agent import app as graph
from langsmith.evaluation import evaluate
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import uuid

thread_id = str(uuid.uuid4())
experiment_prefix = "sql-agent-llama31"
metadata = "chinook-llama3.1-base-case-agent"
dataset_id = "6009592b-7361-4ac5-b6ef-93f852f064ed"

config = {
    "configurable": {
        "thread_id": thread_id,
    }
}

def predict_sql_agent_answer(example: dict):
    msg = {"messages": [("user", example["input"])]}
    result = graph.invoke(msg, config=config)
    return {"response": result["messages"][-1].content}

grade_prompt_answer_accuracy = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a strict evaluator. Compare the student's answer to the reference answer. "
        "Return a single field named Score with value 1 if the answer is correct or semantically equivalent, otherwise 0."
    ),
    (
        "human",
        "Question: {question}\nReference answer: {correct_answer}\nStudent answer: {student_answer}"
    ),
])

class GradeScore(BaseModel):
    Score: int = Field(description="1 if correct, else 0")

llm = ChatOllama(model="llama3.1", temperature=0)
grader = grade_prompt_answer_accuracy | llm.with_structured_output(GradeScore)

def answer_evaluator(run, example) -> dict:
    input_question = example.inputs["input"]
    reference = example.outputs["output"]
    prediction = run.outputs["response"]

    score = grader.invoke({
        "question": input_question,
        "correct_answer": reference,
        "student_answer": prediction,
    })

    return {
        "key": "answer_v_reference_score",
        "score": score.Score,
    }

experiment_results = evaluate(
    predict_sql_agent_answer,
    data=dataset_id,
    evaluators=[answer_evaluator],
    num_repetitions=1,
    max_concurrency=1,
    experiment_prefix=experiment_prefix,
    metadata={"version": metadata},
)

View the evaluation results for experiment: 'sql-agent-llama31-082b82f0' at:
https://smith.langchain.com/o/6d4bdaec-72d5-480a-8ecb-27962e000fe7/datasets/6009592b-7361-4ac5-b6ef-93f852f064ed/compare?selectedSessions=35158c27-8cdb-4ca8-bb16-b0fbff2d6a20




8it [10:32, 79.00s/it] 


## P:240 Testing a single step of an agent

In [13]:
from my_agent.assistant import assistant_runnable
from langsmith.evaluation import evaluate
from langsmith.schemas import Example, Run

dataset_id = "6009592b-7361-4ac5-b6ef-93f852f064ed"
experiment_prefix = "sql-agent-llama31"
metadata = "chinook-llama3.1-base-case-agent"

"""
Single tool evaluation
"""

def predict_assistant(example: dict):
    """Invoke assistant for single tool call evaluation."""
    msg = [("user", example["input"])]
    result = assistant_runnable.invoke({"messages": msg})
    return {"response": result}

def check_specific_tool_call(root_run: Run, example: Example) -> dict:
    """
    Check if the first tool call in the response matches the expected tool call.
    """
    expected_tool_call = "sql_db_list_tables"
    response = root_run.outputs["response"]

    try:
        tool_call = getattr(response, "tool_calls", [])[0]["name"]
    except (IndexError, KeyError, TypeError):
        tool_call = None

    score = 1 if tool_call == expected_tool_call else 0
    return {
        "key": "single_tool_call",
        "score": score,
    }

experiment_results = evaluate(
    predict_assistant,
    data=dataset_id,
    evaluators=[check_specific_tool_call],
    experiment_prefix=experiment_prefix + "-single-tool",
    num_repetitions=1,
    max_concurrency=1,
    metadata={"version": metadata},
)

View the evaluation results for experiment: 'sql-agent-llama31-single-tool-2ae46967' at:
https://smith.langchain.com/o/6d4bdaec-72d5-480a-8ecb-27962e000fe7/datasets/6009592b-7361-4ac5-b6ef-93f852f064ed/compare?selectedSessions=cc2a4404-890b-45b9-bf77-54f155e25e31




5it [00:15,  3.01s/it]


## Testing an agent trajectory

In [14]:
from my_agent.agent import app as graph
from langsmith.evaluation import evaluate
from langsmith.schemas import Example, Run
import uuid

thread_id = str(uuid.uuid4())
experiment_prefix = "sql-agent-llama31"
metadata = "chinook-llama3.1-base-case-agent"
dataset_id = "6009592b-7361-4ac5-b6ef-93f852f064ed"

config = {
    "configurable": {
        "thread_id": thread_id,
    }
}

"""
Agent trajectory evaluation
"""

def predict_sql_agent_messages(example: dict):
    """Run the full graph and return all messages for trajectory evaluation."""
    msg = {"messages": [("user", example["input"])]}
    result = graph.invoke(msg, config=config)
    return {"response": result}

def find_tool_calls(messages):
    """Extract all tool call names from the returned graph messages."""
    tool_calls = [
        tc["name"]
        for m in messages["messages"]
        for tc in getattr(m, "tool_calls", []) or []
    ]
    return tool_calls

EXPECTED_TOOLS = [
    "sql_db_list_tables",
    "sql_db_schema",
    "sql_db_query_checker",
    "sql_db_query",
    "check_result",
]

def contains_all_tool_calls_any_order(root_run: Run, example: Example) -> dict:
    """Check whether all expected tools appear, regardless of order."""
    messages = root_run.outputs["response"]
    tool_calls = find_tool_calls(messages)
    score = 1 if set(EXPECTED_TOOLS).issubset(set(tool_calls)) else 0
    return {"score": score, "key": "multi_tool_call_any_order"}

def contains_all_tool_calls_in_order(root_run: Run, example: Example) -> dict:
    """Check whether all expected tools appear in the expected relative order."""
    messages = root_run.outputs["response"]
    tool_calls = find_tool_calls(messages)

    expected_idx = 0
    for tool in tool_calls:
        if expected_idx < len(EXPECTED_TOOLS) and tool == EXPECTED_TOOLS[expected_idx]:
            expected_idx += 1

    score = 1 if expected_idx == len(EXPECTED_TOOLS) else 0
    return {"score": score, "key": "multi_tool_call_in_order"}

def contains_all_tool_calls_in_order_exact_match(root_run: Run, example: Example) -> dict:
    """Check whether tool calls match the expected sequence exactly, with no extras."""
    messages = root_run.outputs["response"]
    tool_calls = find_tool_calls(messages)
    score = 1 if tool_calls == EXPECTED_TOOLS else 0
    return {"score": score, "key": "multi_tool_call_in_exact_order"}

experiment_results = evaluate(
    predict_sql_agent_messages,
    data=dataset_id,
    evaluators=[
        contains_all_tool_calls_any_order,
        contains_all_tool_calls_in_order,
        contains_all_tool_calls_in_order_exact_match,
    ],
    experiment_prefix=experiment_prefix + "-trajectory",
    num_repetitions=1,
    max_concurrency=1,
    metadata={"version": metadata},
)

View the evaluation results for experiment: 'sql-agent-llama31-trajectory-4a5839ef' at:
https://smith.langchain.com/o/6d4bdaec-72d5-480a-8ecb-27962e000fe7/datasets/6009592b-7361-4ac5-b6ef-93f852f064ed/compare?selectedSessions=94bb3a89-9fea-433c-98f2-e7644ec2a6e5




5it [00:49,  9.97s/it]
